# 1. 청구항 작성 데이터셋 만들기 시도 - 0508(금)
### txt 정제 -> 청구항 추출 -> 가상의 상담 note 만들기 -> (가상의 상담노트, 청구항) pair 만들어 보았음

In [ ]:
import os
import re
import json
import openai
from openai import OpenAI
from pathlib import Path
from google.colab import drive

In [ ]:
client = OpenAI(
    api_key="키"
)

In [ ]:
# ==========================================
# 0. 설정 및 API 키
# ==========================================
# 코랩 환경에서 구글 드라이브를 연결(마운트)합니다.
drive.mount('/content/drive')

OPENAI_API_KEY  = "키"

# [수정 1] 알려주신 구글 드라이브 폴더 구조 반영
# '내 드라이브' 최상단에 'final' 폴더가 있다고 가정합니다.
BASE_DRIVE_PATH = "/content/drive/MyDrive/final/extracted_texts"
CATEGORIES = ['G06N', 'G06F', 'G06V', 'G06Q']

# 결과물(JSONL)도 나중에 다운받기 쉽게 드라이브의 final 폴더에 저장합니다.
OUTPUT_JSONL = "/content/drive/MyDrive/final/patent_finetuning_v1.jsonl"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import re

def clean_text(text):
    """
    단락 번호([0018]), 페이지 구분, 등록특허 번호, 페이지 번호 등을 제거하고
    텍스트를 AI 학습용으로 깔끔하게 다듬습니다.
    """
    # 1. 단락 번호 제거 (예: [0018])
    text = re.sub(r'\[\d{4}\]', '', text)

    # 2. 페이지 구분선 제거 (예: --- [페이지 구분] ---)
    # 하이픈(-) 개수나 띄어쓰기가 유동적일 수 있으므로 정규식으로 묶어버립니다.
    text = re.sub(r'-+\s*\[페이지\s*구분\]\s*-+', '', text)

    # 3. 등록특허 번호 제거 (예: 등록특허 10-2951164)
    # '등록특허' 글자와 그 뒤에 오는 '숫자-숫자' 패턴을 날립니다.
    text = re.sub(r'등록특허\s*\d{2}-\d+', '', text)

    # 4. 페이지 번호 제거 (예: - 3 -)
    # 하이픈 기호 사이에 숫자가 있는 패턴을 날립니다.
    text = re.sub(r'-\s*\d+\s*-', '', text)

    # 5. 불필요한 공백 및 빈 줄 정리
    # 위 항목들을 지우고 나면 엔터(\n)가 여러 개 겹쳐서 빈 공간이 붕 뜨게 되는데,
    # 2번 이상 연속된 줄바꿈을 하나의 줄바꿈(\n)으로 압축합니다.
    text = re.sub(r'\n\s*\n', '\n', text)

    return text.strip()

In [ ]:
def extract_patent_elements(text):
    # 1. 발명의 내용 섹션 추출 (괄호 없이 줄바꿈 기준으로 탐색)
    problem_match = re.search(r'해결하려는\s*과제\s*\n(.*?)(?=\n과제의\s*해결\s*수단)', text, re.S)
    solution_match = re.search(r'과제의\s*해결\s*수단\s*\n(.*?)(?=\n발명의\s*효과)', text, re.S)
    effects_match = re.search(r'발명의\s*효과\s*\n(.*?)(?=\n도면의\s*간단한\s*설명|\n발명을\s*실시하기|\n부호의\s*설명|\Z)', text, re.S)

    problem = clean_text(problem_match.group(1)) if problem_match else ""
    solution = clean_text(solution_match.group(1)) if solution_match else ""
    effects = clean_text(effects_match.group(1)) if effects_match else ""

    # 2. 청구범위 추출 (청구범위 시작부터 발명의 설명 시작 전까지)
    claims = []
    claim_section_match = re.search(r'청구범위\s*\n(.*?)(?=\n발명의\s*설명|\n발명의\s*내용|\n발명의\s*상세한\s*설명)', text, re.S)

    if claim_section_match:
        claims_text = claim_section_match.group(1)
        # '청구항 1', '청구항 2' 등을 기준으로 텍스트를 자름
        chunks = re.split(r'\n청구항\s*\d+\s*\n', '\n' + claims_text)

        for chunk in chunks[1:]: # 첫 번째 빈 조각 제외
            cleaned_chunk = clean_text(chunk)
            if cleaned_chunk and cleaned_chunk != '삭제': # '삭제'된 청구항 무시
                claims.append(cleaned_chunk)

    return {
        "problem": problem,
        "solution": solution,
        "effects": effects,
        "claims": claims
    }

In [ ]:
def synthesize_note(elements):
    # 모델이 역할을 더 잘 수행하도록 System 프롬프트 분리
    system_prompt = "당신은 베테랑 변리사입니다. 발명가와의 가상 '상담 Note'를 전문적이고 논리정연하게 작성해 주세요."

    user_prompt = f"""
다음 특허 명세서의 내용을 바탕으로 상담 Note를 작성하세요.
형식은 반드시 아래 4가지 항목을 포함해야 합니다:
[기존 발명 문제점]
[발명 전체 흐름]
[세부 주안점]
[발명의 효과]

명세서 데이터:
- 해결과제: {elements['problem']}
- 해결수단: {elements['solution']}
- 효과: {elements['effects']}
    """

    try:
        # 최신 API 호출 문법 (client.chat.completions.create)
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.7
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"API Error: {e}"

In [ ]:
import re

def build_consultation_dataset(claims_list, consultation_note):
    """
    상담 Note를 포함하여 전문 변리사 스타일의 멀티턴 데이터셋을 구축합니다.
    (최종 수정: 엔터/공백 무시 로직 적용 및 카테고리 추출 순서 고정)
    """
    # 1. 카테고리별 설정
    categories = {
        '방법': {
            'end_keywords': ['방법'],
            'replace_text': '방법에 있어서',
            'indep_query': "발명의 시계열적 흐름이나 핵심 로직을 담은 '방법' 독립항을 설계하세요."
        },
        '시스템': {
            'end_keywords': ['시스템', '장치'],
            'replace_text': '시스템에 있어서',
            'indep_query': "장치의 구성 요소와 유기적 결합 관계를 바탕으로 '시스템' 독립항을 설계하세요."
        },
        '매체': {
            'end_keywords': ['매체', '기록매체', '기록 매체'],
            'replace_text': '기록 매체에 있어서',
            'indep_query': "방법 발명을 소프트웨어적으로 구현하여 배포하기 위한 '저장매체' 독립항을 설계하세요."
        }
    }

    # 2. 카테고리 추출 순서 강제 고정 (방법 -> 시스템 -> 매체)
    category_order = ['방법', '시스템', '매체']

    citation_pattern = r'^제?\s*\d+\s*항\s*(?:(?:내지|또는|,|및)\s*제?\s*\d+\s*항)*\s*(?:중\s*어느\s*한\s*항)?에\s*있어서,?'
    system_msg = "당신은 전문 변리사입니다. 제공된 상담 Note를 바탕으로 필수 구성요소만을 포함하여 보호 범위를 극대화한 특허 청구항을 작성해야 합니다."

    final_dataset = []

    # 3. 고정된 순서대로 데이터셋 추출
    for cat_key in category_order:
        info = categories[cat_key]

        # [수정된 필터링 로직]
        # 엔터, 띄어쓰기 등 모든 공백을 제거한 순수 텍스트 상태에서 검사합니다.
        cat_claims = []
        for c in claims_list:
            # \s+ 로 모든 공백/엔터 제거 후 마침표 떼기
            clean_c_for_check = re.sub(r'\s+', '', c).rstrip('.')

            # 끝부분 10글자 이내에 키워드가 있는지 확인
            if any(kw in clean_c_for_check[-10:] for kw in info['end_keywords']):
                cat_claims.append(c) # 원본 데이터는 그대로 보존

        indep_claims = []
        dep_claims = []

        for claim in cat_claims:
            if not re.search(citation_pattern, claim.strip()):
                indep_claims.append(claim.strip())
            else:
                # 종속항 문구 수정
                modified = re.sub(citation_pattern, info['replace_text'], claim.strip()).strip()
                modified = re.sub(r'^,\s*', '', modified)
                dep_claims.append(modified)

        # 독립항이 있어야 데이터셋 구성 가능
        if indep_claims:
            # 1턴: 독립항 설계 (상담 Note 포함)
            user_indep = f"다음 상담 Note를 분석하여 {info['indep_query']}\n\n[상담 Note]\n{consultation_note}"
            assistant_indep = "\n\n".join(indep_claims)

            turn_data = {
                "system": system_msg,
                "turns": [
                    {"user": user_indep, "assistant": assistant_indep}
                ]
            }

            # 2턴: 종속항 설계
            if dep_claims:
                user_dep = f"완벽합니다. 이제 앞서 작성한 '{cat_key}' 독립항의 권리범위를 다층적으로 보호하고 구체화하기 위한 종속항들을 작성하세요."
                assistant_dep = "\n\n".join(dep_claims)
                turn_data["turns"].append({"user": user_dep, "assistant": assistant_dep})

            final_dataset.append(turn_data)

    return final_dataset

In [ ]:
# G06V 폴더의 첫 번째 텍스트 파일을 가져와서 테스트합니다.
test_folder = Path(BASE_DRIVE_PATH) / "G06V"
test_files = list(test_folder.glob("*.txt"))

if test_files:
    test_file_path = test_files[0]
    print(f"테스트 파일: {test_file_path.name}\n" + "="*50)

    with open(test_file_path, 'r', encoding='utf-8') as f:
        sample_text = f.read()

    extracted = extract_patent_elements(sample_text)

    print("\n💡 [해결하려는 과제]:\n")
    print(extracted['problem'])
    print("\n" + "-"*50)

    print("\n💡 [과제의 해결 수단]:\n")
    print(extracted['solution'])
    print("\n" + "-"*50)

    print("\n💡 [발명의 효과]:\n")
    print(extracted['effects'])
    print("\n" + "="*50)

    print(f"\n📌 [추출된 청구항 총 {len(extracted['claims'])}개]")
    for i, claim in enumerate(extracted['claims']):
        print(f"\n▶ [청구항 {i+1}]:\n{claim}")
else:
    print("경로에 텍스트 파일이 없습니다. 폴더 경로를 확인해 주세요.")

테스트 파일: 1020250172807.txt

💡 [해결하려는 과제]:

따라서, 본 발명의 제1 목적은 차량의 현장의 영상데이터를 게이트웨이를 통해 클라우드로 전송하고, 클라우드
서버에서 스트립 기반의 이벤트 트리거 전처리 기술과, 번호판의 형태적 특징에 최적화된 딥러닝 아키텍처를 유
기적으로 결합하여, 다중 채널 환경에서의 연산 효율과 인식 정확도를 동시에 대폭 향상시킨 번호판 인식 시스
템을 제공하는데 있다.
 또한, 본 발명의 제2 목적은 영상데이터의 프레임에서 스트립 영역을 설정하고, 스트립 영역의 신호 변화량만
을 분석하는 경량 연산을 수행하여 딥러닝 추론이 필요한 이벤트 프레임을 선별하며, 선별된 이벤트 프레임만
번호판의 문자를 인식하여 전체 연산 부하를 저감하고 다중 채널 처리량을 증대시키는 번호판 인식 방법을 제공
하는데 있다.

--------------------------------------------------

💡 [과제의 해결 수단]:

상술한 본 발명의 제1 목적을 달성하기 위하여, 본 발명의 일 실시예에서는 주행 중인 차량의 영상을 촬영해 영
상데이터를 생성하는 감시 카메라와, 상기 영상데이터를 통신 네트워크를 통해 전송하는 게이트웨이, 및 상기
게이트웨이로부터 수신한 영상데이터의 프레임에서 스트립 영역을 설정하고, 상기 스트립 영역의 신호 변화량을
분석해 차량이 통과하는 이벤트 프레임을 감지해 추출하는 트리아지 코어, 및 번호판 문자열의 수평적 특징과
수직적 특징을 동시에 강조하도록 설계된 딥러닝 모델을 이용하여 상기 이벤트 프레임 내의 번호판 문자를 검지
하고 인식하는 코그니션 코어가 포함된 클라우드 서버를 포함하는 클라우드 기반 전후방 번호판 인식 시스템을
제공한다.
또한, 본 발명의 제2 목적을 달성하기 위하여, 본 발명의 일 실시예에서는 클라우드 서버에서 다중 채널의 영상
데이터를 처리하여 번호판을 인식하는 방법에 있어서, 감시 카메라 및 게이트웨이로부터 제공된 각 채널의 영상
데이터에 대해, 클라우드 서버의 트리아지

In [ ]:
if test_files:
    test_file_path = test_files[0]
    print(f"📄 테스트 파일: {test_file_path.name}")
    print("⏳ 1단계: 텍스트에서 명세서 데이터를 추출합니다...")

    # 파일 읽기
    with open(test_file_path, 'r', encoding='utf-8') as f:
        sample_text = f.read()

    # 데이터 추출
    extracted_data = extract_patent_elements(sample_text)

    # 추출이 잘 되었는지 체크 (빈 데이터면 OpenAI 호출 안 함)
    if not extracted_data['problem']:
        print("❌ 추출 실패: '해결하려는 과제'를 찾지 못했습니다. 텍스트 구조를 확인해주세요.")
    else:
        print("✅ 추출 완료! OpenAI API를 호출하여 상담 Note를 생성합니다...")
        print("=" * 50)

        # OpenAI 합성 실행
        final_note = synthesize_note(extracted_data)

        # 최종 결과 출력
        print(final_note)
        print("=" * 50)
else:
    print("❌ 경로에 텍스트 파일이 없습니다. 구글 드라이브 마운트 및 폴더 경로를 확인해 주세요.")

📄 테스트 파일: 1020250172807.txt
⏳ 1단계: 텍스트에서 명세서 데이터를 추출합니다...
✅ 추출 완료! OpenAI API를 호출하여 상담 Note를 생성합니다...
### 상담 Note

#### [기존 발명 문제점]
기존의 차량 번호판 인식 시스템은 현장에서 고부하의 연산을 수행하기 위해 비싼 현장 제어기를 설치해야 했습니다. 이로 인해 초기 도입 비용 및 운영 비용이 증가하였고, 다양한 환경적 요인(역광, 비, 오염 등)에서도 번호판 인식의 정확성이 떨어지는 문제가 있었습니다. 또한, 다중 채널의 영상 데이터를 처리하는 과정에서 연산 부하가 증가하여 전체 처리량이 저하되는 단점이 존재했습니다.

#### [발명 전체 흐름]
본 발명은 차량의 현장 영상 데이터를 게이트웨이를 통해 클라우드로 전송하고, 클라우드 서버에서 스트립 기반의 이벤트 트리거 전처리 기술과 최적화된 딥러닝 아키텍처를 결합하여 번호판 인식을 수행하는 시스템을 제안합니다. 구체적으로, 감시 카메라가 영상 데이터를 생성하고, 게이트웨이가 이를 클라우드 서버로 전송합니다. 클라우드 서버의 트리아지 코어는 영상 데이터에서 이벤트 프레임을 선별하고, 코그니션 코어가 이 프레임에 대해 번호판 문자를 인식합니다. 이를 통해 불필요한 연산을 줄이고, 효율적으로 번호판 인식을 수행합니다.

#### [세부 주안점]
1. **스트립 영역 설정 및 경량 연산:** 클라우드 서버의 트리아지 코어는 영상 데이터의 프레임에서 스트립 영역을 설정하고, 해당 영역의 신호 변화량을 분석하여 이벤트 프레임을 선별합니다. 이 경량 연산을 통해 딥러닝 추론이 필요한 프레임을 효율적으로 찾아냅니다.
   
2. **딥러닝 모델 최적화:** 코그니션 코어는 번호판의 수평적 및 수직적 특징을 동시에 강조하도록 설계된 딥러닝 모델을 사용하여, 다양한 환경 조건에서도 높은 정확도로 번호판을 검출합니다.

3. **클라우드 기반 처리:** 모든 고부하 연산을 클라우드 서버에서 중앙 집중 처리하여, 현장에서는 감시 카

In [ ]:
print("\n⏳ 2단계: 멀티턴 데이터셋을 구축 중입니다...")
dataset = build_consultation_dataset(extracted_data['claims'], final_note)

print(f"✅ 구축 완료! 생성된 데이터셋 수: {len(dataset)}개 (카테고리별 그룹)")
print("=" * 50)

# 첫 번째 데이터셋 샘플 출력
if dataset:
    sample = dataset[0]
    print(f"📌 [SYSTEM]: {sample['system']}")
    for i, turn in enumerate(sample['turns']):
        print(f"\n[TURN {i+1} USER]: {turn['user'][:100]}...") # 너무 기니까 앞부분만
        print(f"\n[TURN {i+1} AGENT]: {turn['assistant'][:100]}...")
print("=" * 50)


⏳ 2단계: 멀티턴 데이터셋을 구축 중입니다...
✅ 구축 완료! 생성된 데이터셋 수: 2개 (카테고리별 그룹)
📌 [SYSTEM]: 당신은 전문 변리사입니다. 제공된 상담 Note를 바탕으로 필수 구성요소만을 포함하여 보호 범위를 극대화한 특허 청구항을 작성해야 합니다.

[TURN 1 USER]: 다음 상담 Note를 분석하여 발명의 시계열적 흐름이나 핵심 로직을 담은 '방법' 독립항을 설계하세요.

[상담 Note]
### 상담 Note

#### [기존 발명 문제점]
기...

[TURN 1 AGENT]: 클라우드 서버에서 다중 채널의 영상데이터를 처리하여 번호판을 인식하는 방법에 있어서, 
감시 카메라로부터 제공된 각 채널의 영상데이터에 대해, 클라우드 서버의 트리아지 코어가 상기...

[TURN 2 USER]: 완벽합니다. 이제 앞서 작성한 '방법' 독립항의 권리범위를 다층적으로 보호하고 구체화하기 위한 종속항들을 작성하세요....

[TURN 2 AGENT]: 방법에 있어서 상기 프레임 선별단계는
상기 프레임의 해상도를 축소하는 해상도 축소과정과,
해상도가 축소된 프레임에서 차량이 통행하는 영역에 감지선을 도입하는 감지선 도입과정과,
상...


In [ ]:
print(f"DEBUG: 추출된 첫 번째 청구항 예시: {extracted_data['claims'][0] if extracted_data['claims'] else '없음'}")

DEBUG: 추출된 첫 번째 청구항 예시: 주행 중인 차량의 영상을 촬영해 영상데이터를 생성하는 감시 카메라; 및
상기 감시 카메라로부터 제공된 영상데이터의 프레임에서 스트립 영역을 설정하고, 상기 스트립 영역의 신호 변
화량을 분석해 차량이 통과하는 이벤트 프레임을 감지해 추출하는 트리아지 코어, 및 번호판 문자열의 수평적
특징과 수직적 특징을 동시에 강조하도록 설계된 딥러닝 모델을 이용하여 상기 이벤트 프레임 내의 번호판 문자
를 검지하고 인식하는 코그니션 코어가 포함된 클라우드 서버를 포함하며,
상기 트리아지 코어는 상기 스트립 영역의 신호 변화량에 기반한 이벤트 감지 동작과 독립적으로, 이벤트 감지
누락에 대한 안정성을 확보하도록 사전에 설정된 주기에 따라 주기적으로 이벤트 프레임을 강제 추출하여 상기
코그니션 코어로 제공하는 주기적 샘플링 기능을 포함하는 클라우드 기반 전후방 번호판 인식 시스템.


In [ ]:
import json
import time
from pathlib import Path

# 1. Google Drive 마운트 (코랩 환경인 경우 실행)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Colab 환경이 아닙니다. 로컬 경로를 사용합니다.")

# 2. 경로 설정
# 드라이브 최상단(MyDrive) 하위에 final 폴더가 있다고 가정합니다.
# 실제 경로가 다를 경우 '/content/drive/MyDrive/...' 부분을 알맞게 수정해 주세요.
BASE_DIR = Path('/content/drive/MyDrive/final/extracted_texts')
TARGET_FOLDERS = ['G06F', 'G06N', 'G06Q', 'G06V']

# 3. 결과물이 저장될 JSONL 파일 경로
output_jsonl_path = Path('/content/drive/MyDrive/final/patent_multiturn_dataset.jsonl')

print("🚀 데이터셋 추출 및 JSONL 저장 프로세스를 시작합니다...")

# 기존 파일 덮어쓰기 방지를 위해 파일을 새로 생성(초기화)합니다.
# 만약 중간에 끊겨서 이어서 작업해야 한다면 이 부분(블록)을 주석 처리하세요.
with open(output_jsonl_path, 'w', encoding='utf-8') as f:
    pass

total_saved_items = 0

for folder_name in TARGET_FOLDERS:
    folder_path = BASE_DIR / folder_name

    if not folder_path.exists():
        print(f"⚠️ 폴더를 찾을 수 없습니다: {folder_path}")
        continue

    txt_files = list(folder_path.glob("*.txt"))
    print(f"\n📂 [{folder_name}] 폴더 처리 시작 (총 {len(txt_files)}개 파일 발견)")

    for file_path in txt_files:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                text = f.read()

            # 1. 데이터 추출
            extracted = extract_patent_elements(text)
            if not extracted['claims']:
                continue # 청구항이 추출되지 않은 파일은 스킵

            # 2. 상담 Note 생성 (OpenAI API 호출)
            # 💡 중요: OpenAI API Rate Limit 방지를 위해 요청 사이에 약간의 휴식을 줍니다.
            time.sleep(0.5)
            note = synthesize_note(extracted)

            # API 에러 발생 시 건너뛰기
            if "API Error" in note:
                print(f"❌ API 에러 발생 ({file_path.name}) - 스킵합니다.")
                continue

            # 3. 멀티턴 데이터셋 변환
            dataset = build_consultation_dataset(extracted['claims'], note)

            # 4. JSONL 파일에 실시간으로 기록 (Append 모드 'a')
            if dataset:
                with open(output_jsonl_path, 'a', encoding='utf-8') as outfile:
                    for data_item in dataset:
                        # ensure_ascii=False: 한글이 유니코드(예: \uc548)로 깨지는 것을 방지
                        json_line = json.dumps(data_item, ensure_ascii=False)
                        outfile.write(json_line + '\n')
                        total_saved_items += 1

        except Exception as e:
            print(f"⚠️ 파일 처리 중 예기치 않은 오류 발생 ({file_path.name}): {e}")

print("\n" + "="*50)
print(f"✅ 모든 작업 완료! 총 {total_saved_items}개의 학습 데이터가 성공적으로 구축되었습니다.")
print(f"📁 최종 파일 저장 위치: {output_jsonl_path}")
print("="*50)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 데이터셋 추출 및 JSONL 저장 프로세스를 시작합니다...

📂 [G06F] 폴더 처리 시작 (총 110개 파일 발견)

📂 [G06N] 폴더 처리 시작 (총 143개 파일 발견)

📂 [G06Q] 폴더 처리 시작 (총 192개 파일 발견)

📂 [G06V] 폴더 처리 시작 (총 97개 파일 발견)

✅ 모든 작업 완료! 총 895개의 학습 데이터가 성공적으로 구축되었습니다.
📁 최종 파일 저장 위치: /content/drive/MyDrive/final/patent_multiturn_dataset.jsonl
